# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Below, we display available record sets, and for each record set, the fields (columns) and their `@id`.

In [ ]:
# List all record sets and fields with their @id
record_set_ids = [r['@id'] for r in dataset.metadata.to_json().get('recordSet', [])]

print('Available Record Sets (by @id):')
for rs in dataset.metadata.to_json().get('recordSet', []):
    print(f" - {rs['@id']} (type: {rs.get('@type', 'N/A')})")

    # Show field ids and labels in this record set
    fields = rs.get('field', [])
    if fields:
        print('   Fields:')
        # Normalize dict/list
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            # f may be a string or dict (string if just @id, dict if detailed)
            if isinstance(f, str):
                print(f"    - {f}")
            elif isinstance(f, dict):
                print(f"    - {f.get('@id')} : {f.get('name', '')}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. All references use entity `@id`s found above.

> **Note:** For this dataset, typically a main data record set is present, often something like `cr:KilifiMentalHealthSurveyRecords` or similar. Update variable names as necessary with the `@id` from above.

In [ ]:
# Replace with the discovered record set @id from the overview; fallback to guess if empty
if record_set_ids:
    main_record_set_id = record_set_ids[0]
else:
    # Fallback if recordSet list is empty in metadata for some reason
    main_record_set_id = None

# List for use in batch loading
record_sets_to_load = [main_record_set_id] if main_record_set_id else []
dataframes = {}

for record_set in record_sets_to_load:
    print(f"Loading records from record set: {record_set}")
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)
    print(f"Columns in DataFrame for {record_set}: {dataframes[record_set].columns.tolist()}")
    print(dataframes[record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes.

We'll demonstrate filtering, normalization and grouping using field `psq_score` (Perceived Stress Questionnaire score, if present), which is likely numeric. Substitute with an actual field `@id` from the main data, such as `psq_score`, `phq_9_score`, or `gad_7_score`.

In [ ]:
# Guess likely numeric field name; update as needed (find in printout above)
numeric_field = None
possible_numeric_fields = ['psq_score', 'phq_9_score', 'gad_7_score']
if record_sets_to_load and dataframes:
    for f in possible_numeric_fields:
        if f in dataframes[main_record_set_id].columns:
            numeric_field = f
            break

if numeric_field is not None:
    threshold = 10
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping field, e.g., 'village', 'level_of_education', check which present
    group_field = None
    for candidate in ['village', 'level_of_education', 'gender']:
        if candidate in dataframes[main_record_set_id].columns:
            group_field = candidate
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found. Please check available field names in the DataFrame columns.")

## 5. Visualization
Visualize the distribution of a numeric score such as `psq_score`, `phq_9_score`, or any other available numeric field, and explore relation to demographic fields such as `level_of_education`.

Requires `matplotlib` and/or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None and main_record_set_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(
            x=group_field,
            y=numeric_field,
            data=dataframes[main_record_set_id][[group_field, numeric_field]].dropna()
        )
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field or main record set available for plotting.")

## 6. Conclusion
In this notebook, you have loaded the FAIR² Kilifi Mental Health Survey dataset using the `mlcroissant` library, explored its metadata, and performed basic extraction and exploratory data analysis using only entity `@id` references for complete FAIR compliance and future proofing.

- You learned how to identify and reference record sets and fields via `@id`.
- Loaded and previewed tabular record data.
- Conducted elementary analyses including filtering, normalization, grouping, and visualized numeric field distributions.

Continue exploring by utilizing other fields and entities, extending analyses and visualizations as needed for your research goals.